In [154]:
import cv2 as cv
import numpy as np
import os

In [155]:
yolo_path = "E:\\YOLOv3\\New folder"

In [156]:
classes = []
with open("E:\\YOLOv3\\New folder\\coco.names", "r") as f:
    classes = [line.strip() for line in f.readlines()]

In [157]:
yolo_net = cv.dnn.readNet(os.path.join(yolo_path,"yolov3.weights"), os.path.join(yolo_path, "yolov3.cfg"))

In [158]:
layer_names = yolo_net.getLayerNames()
output_layers = [layer_names[i - 1] for i in yolo_net.getUnconnectedOutLayers()]

In [159]:
image = cv.imread("E:\\image\\New folder\\im.jpg")
height, width, channels = image.shape
height , width, channels

(1322, 2260, 3)

In [160]:
# yolo_net.setPreferableBackend(cv.dnn.DNN_BACKEND_CUDA)
# yolo_net.setPreferableTarget(cv.dnn.DNN_TARGET_CUDA)

In [161]:
img_blob = cv.dnn.blobFromImage(image, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
# img_blob = cv.dnn.blobFromImage(image, 1/255.0, (416, 416), swapRB=True, crop=False)

In [162]:
yolo_net.setInput(img_blob)
outputs = yolo_net.forward(output_layers)
outputs[0][0]

array([3.8831845e-02, 5.1986758e-02, 3.8684458e-01, 1.3825296e-01,
       2.7253312e-08, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e

In [163]:
print(cv.getBuildInformation())


General configuration for OpenCV 4.12.0 =====================================
  Version control:               4.12.0

  Platform:
    Timestamp:                   2025-07-04T16:40:32Z
    Host:                        Windows 10.0.26100 AMD64
    CMake:                       3.24.2
    CMake generator:             Visual Studio 17 2022
    CMake build tool:            C:/Program Files/Microsoft Visual Studio/2022/Enterprise/MSBuild/Current/Bin/amd64/MSBuild.exe
    MSVC:                        1944
    Configuration:               Debug Release
    Algorithm Hint:              ALGO_HINT_ACCURATE

  CPU/HW features:
    Baseline:                    SSE SSE2 SSE3
      requested:                 SSE3
    Dispatched code generation:  SSE4_1 SSE4_2 AVX FP16 AVX2 AVX512_SKX
      SSE4_1 (17 files):         + SSSE3 SSE4_1
      SSE4_2 (1 files):          + SSSE3 SSE4_1 POPCNT SSE4_2
      AVX (9 files):             + SSSE3 SSE4_1 POPCNT SSE4_2 AVX
      FP16 (0 files):            + SSSE3 SS

In [164]:
import cv2 as cv
import numpy as np
import os

# مسیر فایل‌ها
yolo_path = r"E:\YOLOv3\New folder"
image_path = r"E:\image\New folder\im.jpg"

weights_path = os.path.join(yolo_path, "yolov3.weights")
cfg_path = os.path.join(yolo_path, "yolov3.cfg")
names_path = os.path.join(yolo_path, "coco.names")

# بارگذاری کلاس‌ها
with open(names_path, "r") as f:
    classes = [line.strip() for line in f.readlines()]

# بارگذاری شبکه
yolo_net = cv.dnn.readNet(weights_path, cfg_path)

# استفاده از CPU برای جلوگیری از خطای CUDA
yolo_net.setPreferableBackend(cv.dnn.DNN_BACKEND_OPENCV)
yolo_net.setPreferableTarget(cv.dnn.DNN_TARGET_CPU)

# گرفتن نام لایه‌های خروجی
output_layers = yolo_net.getUnconnectedOutLayersNames()

# خواندن تصویر
image = cv.imread(image_path)
height, width, channels = image.shape

# تبدیل رنگ به RGB (اختیاری، برای سازگاری بیشتر)
image_rgb = cv.cvtColor(image, cv.COLOR_BGR2RGB)

# ساخت blob
blob = cv.dnn.blobFromImage(image_rgb, 1/255.0, (416, 416), swapRB=True, crop=False)
yolo_net.setInput(blob)

# اجرای forward pass
outputs = yolo_net.forward(output_layers)

# لیست‌ها
boxes = []
confidences = []
class_ids = []

# آستانه‌ها
confidence_threshold = 0.5
nms_threshold = 0.4

# پردازش خروجی‌ها
for out in outputs:
    for detection in out:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]

        if confidence > confidence_threshold:
            center_x = int(detection[0] * width)
            center_y = int(detection[1] * height)
            w = int(detection[2] * width)
            h = int(detection[3] * height)

            x = int(center_x - w / 2)
            y = int(center_y - h / 2)

            boxes.append([x, y, w, h])
            confidences.append(float(confidence))
            class_ids.append(class_id)

# حذف باکس‌های تکراری با NMS
indexes = cv.dnn.NMSBoxes(boxes, confidences, confidence_threshold, nms_threshold)

print("✅ تعداد اشیاء شناسایی‌شده:", len(indexes))

# رنگ‌ها
colors = np.random.uniform(0, 255, size=(len(classes), 3))

# رسم باکس‌ها
if len(indexes) > 0:
    for i in indexes.flatten():
        x, y, w, h = boxes[i]
        label = str(classes[class_ids[i]])
        confidence = confidences[i]
        color = colors[class_ids[i]]

        cv.rectangle(image, (x, y), (x + w, y + h), color, 2)
        cv.putText(image, f"{label} {confidence:.2f}", (x, y - 10),
                   cv.FONT_HERSHEY_PLAIN, 2, color, 2)
else:
    print("⚠️ هیچ شیئی شناسایی نشد.")

# نمایش تصویر
cv.imshow("YOLOv3 Detection", image)
cv.waitKey(0)
cv.destroyAllWindows()

✅ تعداد اشیاء شناسایی‌شده: 14


In [165]:
print([out.shape for out in outputs])


[(507, 85), (2028, 85), (8112, 85)]


In [166]:
class_ids = []
confidences = []
boxes = []
for out in outputs:
    for detection in out:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]
        
        if np.isnan(detection[0]) or np.isnan(detection[1]) or np.isnan(detection[2]) or np.isnan(detection[3]):
            print("NaN detected:", detection)
            continue

        if confidence > 0.2:
            center_x = int(detection[0] * width)
            center_y = int(detection[1] * height)
            w = int(detection[2] * width)
            h = int(detection[3] * height)

            # Rectangle coordinates
            x = int(center_x - w / 2)
            y = int(center_y - h / 2)
            boxes.append([x, y, w, h])
            confidences.append(float(confidence))
            class_ids.append(class_id)
            

In [167]:
colors = np.random.uniform(0, 255, size=(len(classes), 3))
for i in range(len(boxes)):
    x, y, w, h =boxes[i]
    label = str(classes[class_ids[i]])
    color = colors[class_ids[i]]
    cv.rectangle(image, (x,y), (x + w, y + h), color, 2)
    cv.putText(image, label, (x, y + 30), cv.FONT_HERSHEY_PLAIN, 1, color, 1)

cv.imwrite("E:\\image\\New folder\\im1.jpg",image)

True

In [168]:
indexes = cv.dnn.NMSBoxes(boxes, confidences, 0.3, 0.4)

for i in indexes:
    i = i[0] if isinstance(i, (list, np.ndarray)) else i
    x, y, w, h = boxes[i]
    label = str(classes[class_ids[i]])
    color = colors[class_ids[i]]
    cv.rectangle(image, (x, y), (x + w, y + h), color, 2)
    cv.putText(image, label, (x, y - 10), cv.FONT_HERSHEY_PLAIN, 2, color, 2)


In [169]:
cv.imshow("Image", image)
cv.waitKey(0)
cv.destroyAllWindows()

In [170]:
with open(cfg_path, "r") as f:
    for _ in range(10):
        print(f.readline())


[net]

# Testing

# batch=1

# subdivisions=1

# Training

batch=64

subdivisions=16

width=608

height=608

channels=3

